### Methodology - Extract, Load and Transform
1. Read (From Bronze Table)
2. Transform DATA
3. Load/ Write (into Silver Table)

### TRANSFORMATION WORKFLOW

1. TRIM SPACES
2. NORMALIZE MARITAL_STATUS
3. NORMALIZE GENDER
4. RENAME COLUMNS
5. WRITE INTO SILVER

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

In [0]:
# 0 LOAD DATA & READ FROM BRONZE
# Read From bronze

df = spark.table("acdproj.bronze.crm_cust_info")


In [0]:
# 1) TRIM SPACES
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

In [0]:
# 2) NORMALIZE MARITAL_STATUS
df = df.withColumn(
    "cst_marital_status",
    F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
     .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
     .otherwise("n/a")
)

In [0]:
# 3) NORMALIZE GENDER
df = df.withColumn(
    "cst_gndr",
    F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
     .when(F.upper(F.col("cst_gndr")) == "F", "Female")
     .otherwise("n/a")
)

In [0]:
# 4) RENAME COLUMNS
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "customer_gender",
    "cst_create_date": "create_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
# 5) REMOVE INVALID/JUNK CUSTOMER KEYS
# Some records in the source don't follow the expected AW + 8-digit format
# and have no name data — treated as test/junk records, not real customers
df = df.filter(F.col("customer_key").rlike("^AW\\d{8}$"))

# 6) Write into Silver Table
df.write.mode("overwrite").saveAsTable("acdproj.silver.crm_customers")